# Caso Práctico Unidad 3

## 1. Transfer Learning y Fine Tuning para clasificación de imágenes



1. **¿Cuáles son los motivos principales por los que entrenar un modelo desde 0 no sería lo ideal en este caso?**

Entrenar un modelo desde 0 no es lo ideal principalmente por varias razones: requiere mucho tiempo de entrenamiento, gran capacidad computacional y una enorme cantidad de datos etiquetados (aunque el dataset tiene más de 8000 imágenes en total, están repartidas entre 101 clases, por lo que hay muy pocas imágenes por categoría).


2. **¿Qué tendría que cambiar para que pudiéramos entrenar un modelo desde 0 para resolver este problema?**

Para poder entrenar un modelo desde cero de forma efectiva necesitaríamos muchos más datos de entrenamiento, aumentando considerablemente el número de imágenes por clase; aplicar técnicas de Data Augmentation para generar variaciones artificiales y mejorar la generalización; y, finalmente, mucha más potencia computacional, GPUs más potentes y más tiempo de entrenamiento.


3. **Si quisiéramos un modelo capaz de ser entrenado lo más rápidamente posible a cambio de poder tener un peor rendimiento, ¿cómo lo haríamos en este caso?**

Utilizar únicamente Transfer Learning, congelando completamente las capas convolucionales del modelo VGG16 y entrenando solo la capa clasificadora final, reducir el tamaño de las imágenes de entrada, entrenar durante menos épocas, utilizar un batch size mayor si la memoria lo permite y reducir la complejidad del clasificador final añadiendo menos capas densas.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [6]:
PATH = "drive/MyDrive/Formaciones/Máster DL - IEP/6.- Transfer Learning/dataset/101_ObjectCategories"

In [20]:
import keras
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# Configurar los generadores de datos para entrenamiento y prueba
train_datagen = ImageDataGenerator(rescale=1./255)

test_datagen = ImageDataGenerator(rescale=1./255)  # Normalización de píxeles

# Especificar cómo leer los datos de entrenamiento y prueba desde el directorio
train_generator = train_datagen.flow_from_directory(
        directory= PATH + '/train',
        target_size = (224, 224),  # Tamaño de las imágenes (ajústalo según sea necesario)
        batch_size = 32,
        shuffle=True,
        class_mode = 'categorical')  # Modo de clasificación categórica

test_generator = test_datagen.flow_from_directory(
        directory = PATH + '/val',
        target_size = (224, 224),  # Tamaño de las imágenes (ajústalo según sea necesario)
        batch_size = 16,
        shuffle=True,
        class_mode = 'categorical')  # Modo de clasificación categórica

Found 6907 images belonging to 101 classes.
Found 1770 images belonging to 101 classes.


In [21]:
from keras.models import Sequential
from keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout

# Definir el modelo de la CNN
model = Sequential()
model.add(Conv2D(32, (3, 3), activation='relu', input_shape=(224, 224, 3)))
model.add(MaxPooling2D((2, 2)))
model.add(Conv2D(64, (3, 3), activation='relu'))
model.add(MaxPooling2D((2, 2)))
model.add(Conv2D(64, (3, 3), activation='relu'))
model.add(Flatten())
model.add(Dense(512, activation='relu'))
model.add(Dropout(0.5))
model.add(Dense(101, activation='softmax'))

# Compilar el modelo
model.compile(optimizer='adam',
              loss='categorical_crossentropy',
              metrics=['accuracy'])

# Entrenar el modelo
history = model.fit(
      train_generator,
      steps_per_epoch=train_generator.samples // train_generator.batch_size,
      epochs=20,
      validation_data=test_generator,
      validation_steps=test_generator.samples // test_generator.batch_size)

# Evaluar el modelo
test_loss, test_acc = model.evaluate(test_generator, verbose=2)
print('\nTest accuracy:', test_acc)

Epoch 1/20
215/215 ━━━━━━━━━━━━━━━━━━━━ 2332s 11s/step - accuracy: 0.2752 - loss: 3.5912 - val_accuracy: 0.4051 - val_loss: 2.7966
Epoch 2/20
215/215 ━━━━━━━━━━━━━━━━━━━━ 11s 35ms/step - accuracy: 0.4688 - loss: 2.7723 - val_accuracy: 0.4045 - val_loss: 2.8452
Epoch 3/20
215/215 ━━━━━━━━━━━━━━━━━━━━ 39s 181ms/step - accuracy: 0.4903 - loss: 2.2677 - val_accuracy: 0.4898 - val_loss: 2.3432
Epoch 4/20
215/215 ━━━━━━━━━━━━━━━━━━━━ 8s 36ms/step - accuracy: 0.3750 - loss: 3.2166 - val_accuracy: 0.4892 - val_loss: 2.3566
Epoch 5/20
215/215 ━━━━━━━━━━━━━━━━━━━━ 72s 170ms/step - accuracy: 0.6963 - loss: 1.2255 - val_accuracy: 0.5068 - val_loss: 2.3389
Epoch 6/20
215/215 ━━━━━━━━━━━━━━━━━━━━ 8s 36ms/step - accuracy: 0.5625 - loss: 1.5516 - val_accuracy: 0.5028 - val_loss: 2.3449
Epoch 7/20
215/215 ━━━━━━━━━━━━━━━━━━━━ 37s 170ms/step - accuracy: 0.8537 - loss: 0.5753 - val_accuracy: 0.5080 - val_loss: 2.5099
Epoch 8/20
215/215 ━━━━━━━━━━━━━━━━━━━━ 8s 36ms/step - accuracy: 0.8438 - loss: 0.5272 -

### Transfer learning

In [22]:
from keras.applications import VGG16
from keras.models import Sequential
from keras.layers import Flatten, Dense, Dropout
from keras.optimizers import Adam

# Cargar el modelo VGG16 pre-entrenado sin la capa de clasificación
base_model = VGG16(
    weights='imagenet',
    include_top=False,
    input_shape=(224, 224, 3)
)

# Congelar las capas del modelo base
for layer in base_model.layers:
    layer.trainable = False

# Definir el modelo secuencial sobre el modelo base
model = Sequential([
    base_model,
    Flatten(),
    Dense(512, activation='relu'),
    Dropout(0.5),
    Dense(101, activation='softmax')
])

# Compilar el modelo
model.compile(
    optimizer=Adam(learning_rate=0.0001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# Entrenar el modelo
history = model.fit(
      train_generator,
      steps_per_epoch=train_generator.samples // train_generator.batch_size,
      epochs=20,
      validation_data=test_generator,
      validation_steps=test_generator.samples // test_generator.batch_size)

# Evaluar el modelo
test_loss, test_acc = model.evaluate(test_generator, verbose=2)
print('\nTest accuracy:', test_acc)

Epoch 1/20
215/215 ━━━━━━━━━━━━━━━━━━━━ 65s 291ms/step - accuracy: 0.4935 - loss: 2.4300 - val_accuracy: 0.7455 - val_loss: 1.3519
Epoch 2/20
215/215 ━━━━━━━━━━━━━━━━━━━━ 11s 51ms/step - accuracy: 0.6875 - loss: 1.7880 - val_accuracy: 0.7420 - val_loss: 1.3487
Epoch 3/20
215/215 ━━━━━━━━━━━━━━━━━━━━ 48s 222ms/step - accuracy: 0.7658 - loss: 1.0508 - val_accuracy: 0.8301 - val_loss: 0.8195
Epoch 4/20
215/215 ━━━━━━━━━━━━━━━━━━━━ 21s 95ms/step - accuracy: 0.8438 - loss: 0.9134 - val_accuracy: 0.8301 - val_loss: 0.8223
Epoch 5/20
215/215 ━━━━━━━━━━━━━━━━━━━━ 48s 225ms/step - accuracy: 0.8671 - loss: 0.6085 - val_accuracy: 0.8619 - val_loss: 0.6528
Epoch 6/20
215/215 ━━━━━━━━━━━━━━━━━━━━ 11s 52ms/step - accuracy: 0.6562 - loss: 1.1426 - val_accuracy: 0.8625 - val_loss: 0.6509
Epoch 7/20
215/215 ━━━━━━━━━━━━━━━━━━━━ 48s 223ms/step - accuracy: 0.9191 - loss: 0.3915 - val_accuracy: 0.8773 - val_loss: 0.5667
Epoch 8/20
215/215 ━━━━━━━━━━━━━━━━━━━━ 11s 52ms/step - accuracy: 0.9062 - loss: 0.337

El modelo basado en Transfer Learning con VGG16 ha obtenido resultados muy buenos, alcanzando una precisión final cercana al 90% en el conjunto de test (accuracy ≈ 0.896). Esto demuestra la gran eficacia de utilizar modelos preentrenados en problemas con pocas imágenes por clase y un número elevado de categorías.

Además, puede observarse cómo tanto la accuracy de entrenamiento como la de validación aumentan progresivamente durante las épocas, mientras la pérdida (loss) disminuye de forma estable. La diferencia relativamente pequeña entre accuracy y val_accuracy indica que el modelo generaliza bastante bien y no presenta un sobreajuste excesivo.

Comparado con el modelo entrenado desde cero, el Transfer Learning mejora enormemente el rendimiento, confirmando que reutilizar características aprendidas previamente en ImageNet es mucho más adecuado para este tipo de problema.

### Fine Tuning

In [23]:
from keras.applications import VGG16
from keras.models import Sequential
from keras.layers import Flatten, Dense, Dropout
from keras.optimizers import Adam

# Cargar el modelo VGG16 pre-entrenado sin la capa de clasificación
base_model = VGG16(
    weights='imagenet',
    include_top=False,
    input_shape=(224, 224, 3)
)

# En Fine Tuning NO congelamos las capas del modelo base
for layer in base_model.layers:
    layer.trainable = True

# Definir el modelo secuencial sobre el modelo base
model = Sequential()
model.add(base_model)
model.add(Flatten())
model.add(Dense(512, activation='relu'))
model.add(Dropout(0.5))
model.add(Dense(101, activation='softmax'))

# Compilar el modelo con learning rate bajo
model.compile(
    optimizer=Adam(learning_rate=0.00001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# Entrenar el modelo
history = model.fit(
    train_generator,
    steps_per_epoch=train_generator.samples // train_generator.batch_size,
    epochs=20,
    validation_data=test_generator,
    validation_steps=test_generator.samples // test_generator.batch_size
)

# Evaluar el modelo
test_loss, test_acc = model.evaluate(test_generator, verbose=2)
print('\nTest accuracy:', test_acc)

Epoch 1/20
215/215 ━━━━━━━━━━━━━━━━━━━━ 168s 737ms/step - accuracy: 0.3583 - loss: 3.1338 - val_accuracy: 0.6347 - val_loss: 1.7616
Epoch 2/20
215/215 ━━━━━━━━━━━━━━━━━━━━ 12s 52ms/step - accuracy: 0.6562 - loss: 1.6996 - val_accuracy: 0.6375 - val_loss: 1.7561
Epoch 3/20
215/215 ━━━━━━━━━━━━━━━━━━━━ 117s 542ms/step - accuracy: 0.6640 - loss: 1.4466 - val_accuracy: 0.8080 - val_loss: 0.8941
Epoch 4/20
215/215 ━━━━━━━━━━━━━━━━━━━━ 12s 52ms/step - accuracy: 0.5938 - loss: 1.5730 - val_accuracy: 0.8097 - val_loss: 0.8864
Epoch 5/20
215/215 ━━━━━━━━━━━━━━━━━━━━ 117s 542ms/step - accuracy: 0.8121 - loss: 0.7455 - val_accuracy: 0.8415 - val_loss: 0.6707
Epoch 6/20
215/215 ━━━━━━━━━━━━━━━━━━━━ 12s 52ms/step - accuracy: 0.7812 - loss: 0.6741 - val_accuracy: 0.8432 - val_loss: 0.6643
Epoch 7/20
215/215 ━━━━━━━━━━━━━━━━━━━━ 117s 543ms/step - accuracy: 0.8803 - loss: 0.4460 - val_accuracy: 0.8636 - val_loss: 0.5578
Epoch 8/20
215/215 ━━━━━━━━━━━━━━━━━━━━ 12s 52ms/step - accuracy: 0.8750 - loss: 0

En el caso del Fine Tuning, el modelo también obtiene un rendimiento muy elevado, alcanzando una precisión final cercana al 88% en el conjunto de test (accuracy ≈ 0.883). El entrenamiento muestra una mejora progresiva tanto en accuracy como en pérdida durante las distintas épocas.

Sin embargo, en este caso el rendimiento final es ligeramente inferior al obtenido mediante Transfer Learning puro. Además, se observa que el modelo alcanza valores extremadamente altos de accuracy en entrenamiento (incluso 100%), mientras que la val_accuracy se estabiliza alrededor del 88%, lo que indica la aparición de cierto sobreajuste.

Esto puede deberse a que, al descongelar las capas del modelo VGG16 durante el Fine Tuning, el modelo comienza a adaptarse demasiado al conjunto de entrenamiento, especialmente teniendo en cuenta que el dataset es relativamente pequeño para un problema con 101 clases.

En conclusión, aunque el Fine Tuning permite ajustar más específicamente el modelo al problema concreto, en este caso el Transfer Learning puro ha ofrecido un resultado ligeramente mejor y más estable.

## 2. Prompt Engineering

### Ejercicio 2.1

La técnica principal utilizada en este caso sería One-Shot Prompting.

Consiste en proporcionar al modelo un ejemplo o una estructura concreta de cómo debe generar la oferta comercial, para que posteriormente pueda reutilizar ese mismo patrón cambiando únicamente ciertos datos como perfiles profesionales, duración del proyecto, presupuesto y tecnologías utilizadas.

Gracias a ello, el modelo aprende el formato esperado y puede generar nuevas propuestas manteniendo una estructura coherente y profesional.

### Ejercicio 2.2

El prompt sería el siguiente:

Eres un especialista de recursos humanos encargado de seleccionar candidatos para un puesto de Data Scientist en un proyecto de comercio internacional para la Secretaría de Estado del Comercio.

Analiza el siguiente currículum y extrae únicamente la información relevante para el puesto.

Ignora experiencia no relacionada directamente con:
- análisis de datos,
- programación,
- machine learning,
- Big Data,
- estadística,
- visualización de datos,
- tecnologías de datos,
- habilidades analíticas.

Resume el perfil del candidato destacando:
1. Formación relevante
2. Experiencia relevante
3. Tecnologías y herramientas
4. Habilidades útiles para el puesto
5. Valoración final del candidato para el puesto


Este es el resultado aplicado al CV del notebook:

1. Formación relevante
Máster en Big Data.
Formación adicional en programación e idiomas.
Conocimientos técnicos orientados al análisis y procesamiento de datos.
2. Experiencia relevante

Aunque la mayor parte de la experiencia profesional del candidato pertenece al ámbito legal, destacan varias competencias transferibles útiles para el puesto:

Elaboración de informes y análisis de información compleja.
Resolución de problemas y toma de decisiones.
Colaboración con diferentes departamentos.
Gestión de equipos y coordinación de proyectos.

Además, el candidato muestra una transición profesional hacia el ámbito tecnológico y de datos mediante formación especializada en Big Data.

3. Tecnologías y herramientas
Python
SQL
R
Hadoop
Spark
Herramientas de Big Data
Modelado predictivo
Análisis de datos
4. Habilidades útiles para el puesto
Capacidad analítica avanzada.
Resolución de problemas complejos.
Interpretación y gestión de información.
Comunicación efectiva.
Capacidad de trabajo interdisciplinar.
Adaptabilidad y aprendizaje continuo.
Conocimiento del ámbito regulatorio y comercial, potencialmente útil para proyectos relacionados con comercio internacional.
5. Valoración final del candidato para el puesto

El candidato presenta un perfil híbrido interesante, combinando experiencia profesional en el ámbito legal con una reciente especialización en Big Data y análisis de datos. Aunque su experiencia práctica directa como Data Scientist parece limitada, dispone de conocimientos técnicos relevantes en programación y tecnologías Big Data, además de sólidas capacidades analíticas y de resolución de problemas.

Podría encajar especialmente bien en proyectos relacionados con análisis de datos en contextos regulatorios, jurídicos o de comercio internacional, donde su experiencia legal aporte valor añadido.

**¿Qué tipo de técnicas de prompt engineering se han probado para elaborar el prompt?¿Cuáles han funcionado mejor?**

**One-Shot Prompting**

Se proporcionó una estructura clara indicando exactamente qué información debía extraerse del currículum y qué debía ignorarse.

Esta técnica funcionó especialmente bien porque el problema está bien definido, no requiere razonamientos excesivamente complejos y permite obtener respuestas estructuradas y consistentes.

**Chain-Of-Thought**

También se probó una aproximación donde el modelo razonaba paso a paso sobre qué experiencia era relevante, qué habilidades encajaban con el puesto y qué partes del CV debían descartarse.

Sin embargo, en este caso no aportó una mejora significativa respecto al One-Shot Prompting, ya que la tarea consistía principalmente en filtrar y resumir información. Por lo tanto, la técnica que mejor funcionó fue **One-Shot Prompting**, porque permitió guiar al modelo de forma clara y directa para extraer únicamente la información relevante para el puesto de Data Scientist.